In [2]:
# Name: Shaza Adelina
# ID: SN0107857

In [6]:
# Import libraries
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

#required resources 
nltk.download('stopwords')
nltk.download('wordnet')

# Read dataset
df = pd.read_csv("news_dataset.csv")

# Use only 'text' column
texts = df['text'].dropna()

# Initialise tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


#PREPROCESSING

def preprocess(text):
    text = text.lower()
    
    # keep only letters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # split into words FIRST
    words = text.split()
    
    # filter words properly
    words = [w for w in words if w not in stop_words and len(w) > 3 and len(w) < 15]
    
    # lemmatize
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return words

# Apply preprocessing
processed_texts = [preprocess(text) for text in texts]

# Remove very short documents 
processed_texts = [text for text in processed_texts if len(text) > 5]


# CREATE DICTIONARY & CORPUS

dictionary = corpora.Dictionary(processed_texts)

# Filter extremes
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in processed_texts]


# TRAIN LDA MODEL
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=10
)


print("\n=== FINAL TOPICS ===")

topics = lda_model.show_topics(num_topics=4, num_words=8, formatted=False)

for i, topic in topics:
    words = [word for word, prob in topic]
    print(f"Topic {i}: {words}")


# COHERENCE SCORE
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("\nCoherence Score:", coherence_score)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



=== FINAL TOPICS ===
Topic 0: ['encryption', 'chip', 'system', 'government', 'information', 'clipper', 'file', 'key']
Topic 1: ['game', 'team', 'year', 'university', 'player', 'contact', 'play', 'season']
Topic 2: ['would', 'window', 'like', 'know', 'work', 'problem', 'system', 'also']
Topic 3: ['would', 'people', 'dont', 'think', 'know', 'like', 'right', 'time']

Coherence Score: 0.5030134478508537
